[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-09-benchmarking.ipynb#scrollTo=11a2b3c4)

---
# Day 9 · Benchmarking — Tokens/sec, Memory, and Quality Metrics
**certified-journeys / llama-certified** · Day 9 · Measurement & Evaluation

> **Goal for today:** Write a Python benchmarker that measures prefill latency, decode tokens/sec, and peak RAM across model variants, build an LLM-judge quality evaluator, and produce a one-page benchmark report comparing base vs. fine-tuned Llama.


In [ ]:
%pip install -q psutil tabulate


## What is lm-evaluation-harness?

[EleutherAI's lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) is the de-facto open-source LLM benchmarking framework. It supports 60+ tasks out of the box (HellaSwag, TruthfulQA, MMLU, GSM8K, ARC, etc.) and can evaluate models via HuggingFace, VLLM, or the OpenAI API.

| Concept | Description |
|---|---|
| **Task** | A benchmark dataset + evaluation metric (e.g. accuracy, F1) |
| **Few-shot** | Number of in-context examples provided before the question |
| **Loglikelihood** | Default scoring: pick the answer with highest log P(answer \| context) |
| **Generate-until** | Open-ended generation tasks scored by exact match or ROUGE |

```bash
# Production usage (run outside this notebook):
pip install lm_eval
lm_eval --model hf --model_args pretrained=meta-llama/Llama-3.2-1B \
        --tasks arc_easy,hellaswag --num_fewshot 0 --output_path ./results
```

In this notebook we build a **custom lightweight benchmarker** that measures the same three key signals — prefill latency, decode speed, and peak RAM — without requiring a full GPU environment.


## Step 1 · Understand the Two Inference Phases

Every LLM inference call has two distinct phases with very different performance profiles:

| Phase | What happens | Bottleneck | Scales with |
|---|---|---|---|
| **Prefill** | Process the entire prompt in parallel | Memory bandwidth | Prompt length |
| **Decode** | Generate tokens one at a time (autoregressive) | Compute | Output length |

**Key rule:** measure tokens/sec at the **decode phase**, not prefill. A slow prefill means a long prompt; slow decode means the GPU is the bottleneck.

Diagram:
```
Prompt tokens (N) ──► [PREFILL]  ──► KV cache ──► [DECODE token 1] ──► ... ──► [DECODE token M]
                       ~constant        grows           repeating
```


In [ ]:
import time, random, math

def simulate_prefill(prompt_tokens: int, model_size_b: float = 1.0) -> float:
    """Simulate prefill latency in seconds. Scales linearly with prompt length."""
    # Rough heuristic: 1B model processes ~4000 tok/s in prefill on a modern CPU
    prefill_rate = 4000 / model_size_b   # smaller models are faster
    jitter = random.uniform(0.9, 1.1)
    return (prompt_tokens / prefill_rate) * jitter


def simulate_decode(n_tokens: int, model_size_b: float = 1.0) -> tuple[list[float], float]:
    """Simulate per-token decode latency. Returns (per_token_latencies, peak_ram_gb)."""
    # Rough heuristic: 1B model decodes ~30 tok/s on CPU, ~200 tok/s on GPU
    decode_rate = 30 / model_size_b   # tok/s on CPU
    base_latency = 1.0 / decode_rate  # seconds per token
    latencies = [base_latency * random.uniform(0.85, 1.15) for _ in range(n_tokens)]
    # RAM: roughly 2 bytes/param for Q4 quantization + KV cache
    peak_ram_gb = model_size_b * 0.6 + 0.2  # 0.6 GB/B params (Q4) + overhead
    return latencies, round(peak_ram_gb, 2)


# Demo: profile a single call for llama3.2:1b
prompt_tokens = 64
output_tokens = 50

prefill_t = simulate_prefill(prompt_tokens, model_size_b=1.0)
decode_latencies, peak_ram = simulate_decode(output_tokens, model_size_b=1.0)
decode_total = sum(decode_latencies)
tok_per_sec  = output_tokens / decode_total

print(f'Prefill latency  : {prefill_t*1000:.1f} ms  ({prompt_tokens} prompt tokens)')
print(f'Decode speed     : {tok_per_sec:.1f} tok/s  ({output_tokens} tokens)')
print(f'Decode total     : {decode_total*1000:.0f} ms')
print(f'Peak RAM (model) : {peak_ram:.2f} GB')
print(f'Total wall time  : {(prefill_t + decode_total)*1000:.0f} ms')


**What just happened?**

- Prefill is **fast and roughly constant** for short prompts — it parallelises across the prompt sequence.
- Decode latency **accumulates** with output length — this is what users experience as 'slowness'.
- **Always report decode tok/s**, not total tokens/wall_time — prefill noise contaminates the latter.
- Peak RAM for a Q4 model is approximately `0.5–0.6 GB per billion parameters`.


## Step 2 · Multi-Model Benchmarker

We benchmark three model variants representing a typical fine-tuning experiment:

| Variant | Description | Expected tok/s |
|---|---|---|
| `llama3.2:1b-base` | Unquantised base model | ~20–35 |
| `llama3.2:1b-q4` | Q4_K_M quantised (GGUF) | ~40–80 |
| `llama3.2:1b-ft-q4` | Fine-tuned + Q4_K_M | ~40–80 |

The benchmarker fires multiple prompts per model and averages results.


In [ ]:
import time, random, statistics
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class BenchmarkResult:
    model:            str
    n_runs:           int
    avg_prefill_ms:   float
    avg_decode_tps:   float   # tokens per second
    p99_decode_ms:    float   # 99th percentile per-token latency in ms
    peak_ram_gb:      float
    total_tokens_out: int


# Model configs: (model_name, size_in_billions, is_quantised)
MODEL_CONFIGS = [
    ('llama3.2:1b-base',   1.0,  False),
    ('llama3.2:1b-q4',     1.0,  True),
    ('llama3.2:1b-ft-q4',  1.0,  True),  # fine-tuned, same arch/size
]

BENCHMARK_PROMPTS = [
    'Explain the difference between supervised and unsupervised learning.',
    'What is the attention mechanism in transformer models?',
    'How does quantization reduce model size without retraining?',
    'Describe the QLoRA fine-tuning workflow step by step.',
    'What is the purpose of a Modelfile in Ollama?',
]


def benchmark_model(model_name: str, size_b: float, is_quantised: bool,
                    prompts: list[str], output_tokens: int = 60) -> BenchmarkResult:
    """Run benchmark for one model across all prompts."""
    # Quantised models are ~2x faster to decode
    effective_size = size_b * (0.45 if is_quantised else 1.0)

    prefill_times, all_decode_lats, peak_rams = [], [], []
    total_out = 0

    for prompt in prompts:
        prompt_tokens = len(prompt.split())  # rough token count
        pf = simulate_prefill(prompt_tokens, model_size_b=effective_size)
        dec_lats, ram = simulate_decode(output_tokens, model_size_b=effective_size)
        prefill_times.append(pf * 1000)     # convert to ms
        all_decode_lats.extend(dec_lats)    # per-token latencies in seconds
        peak_rams.append(ram)
        total_out += output_tokens

    total_decode_s = sum(all_decode_lats)
    avg_tps = total_out / total_decode_s
    p99_ms  = sorted(all_decode_lats)[int(len(all_decode_lats) * 0.99)] * 1000

    return BenchmarkResult(
        model=model_name,
        n_runs=len(prompts),
        avg_prefill_ms=round(statistics.mean(prefill_times), 1),
        avg_decode_tps=round(avg_tps, 1),
        p99_decode_ms=round(p99_ms, 1),
        peak_ram_gb=round(max(peak_rams), 2),
        total_tokens_out=total_out,
    )


results = [benchmark_model(name, size, quant, BENCHMARK_PROMPTS)
           for name, size, quant in MODEL_CONFIGS]

print(f'{"Model":<25} {"Prefill ms":>12} {"Tok/s":>8} {"p99 ms":>9} {"RAM GB":>8}')
print('-' * 65)
for r in results:
    print(f'{r.model:<25} {r.avg_prefill_ms:>12.1f} {r.avg_decode_tps:>8.1f} {r.p99_decode_ms:>9.1f} {r.peak_ram_gb:>8.2f}')


**What just happened?**

- We ran each prompt through a simulated inference pipeline, separating prefill and decode timings.
- The Q4 variants show higher tok/s because quantisation reduces the memory footprint the CPU must stream per layer.
- **p99 latency** is more useful than average for SLO decisions — it captures the worst-case token latency.
- Fine-tuned Q4 and base Q4 have similar speed — fine-tuning changes weights but not architecture or quantisation.


## Step 3 · Measure Peak RAM with psutil

On a real machine, track peak RSS (resident set size) before and after loading a model:

```python
import psutil, os
proc = psutil.Process(os.getpid())
ram_before = proc.memory_info().rss / 1e9   # GB
# ... load model ...
ram_after  = proc.memory_info().rss / 1e9
peak_ram   = ram_after - ram_before
```

We simulate this below with realistic values per model variant.


In [ ]:
import psutil, os

proc = psutil.Process(os.getpid())
current_ram_gb = proc.memory_info().rss / 1e9

print(f'Current process RSS: {current_ram_gb:.2f} GB')
print()

# Simulated peak RAM for each variant (values match Q4 formula from Step 1)
EXPECTED_RAM = {
    'llama3.2:1b-base':   {'model_gb': 2.0, 'kv_cache_gb': 0.3, 'overhead_gb': 0.2},
    'llama3.2:1b-q4':     {'model_gb': 0.8, 'kv_cache_gb': 0.2, 'overhead_gb': 0.2},
    'llama3.2:1b-ft-q4':  {'model_gb': 0.8, 'kv_cache_gb': 0.2, 'overhead_gb': 0.2},
}

print(f'{"Model":<25} {"Model GB":>10} {"KV Cache":>10} {"Overhead":>10} {"Total GB":>10}')
print('-' * 65)
for model, m in EXPECTED_RAM.items():
    total = m['model_gb'] + m['kv_cache_gb'] + m['overhead_gb']
    print(f'{model:<25} {m["model_gb"]:>10.2f} {m["kv_cache_gb"]:>10.2f} {m["overhead_gb"]:>10.2f} {total:>10.2f}')

print()
print('Rule of thumb: Q4_K_M quantisation uses ~0.5 GB per billion parameters')
print('Production: use psutil.Process().memory_info().rss before/after model load')


**What just happened?**

- We decomposed peak RAM into three components: model weights, KV cache, and runtime overhead.
- Q4 quantisation cuts model weight memory from ~2 GB to ~0.8 GB for a 1B model — a 2.5x reduction.
- **KV cache** grows with sequence length; cap it by reducing `--ctx-size` in Ollama's Modelfile if RAM is tight.
- For GPU inference, replace `psutil.rss` with `torch.cuda.max_memory_allocated()` or `nvidia-smi`.


## Step 4 · Build a Quality Evaluator with an LLM Judge

Speed metrics alone do not tell you whether the fine-tuned model is **better**.  
An LLM judge scores each output on a 1–5 scale:

| Score | Meaning |
|---|---|
| 1 | Irrelevant or hallucinated |
| 2 | Partially correct but missing key points |
| 3 | Adequate — answers the question |
| 4 | Good — accurate and reasonably detailed |
| 5 | Excellent — accurate, concise, well-structured |

The judge prompt follows the **G-Eval** format: provide criteria, context, response, and ask for a numeric score.


In [ ]:
import random

# Evaluation prompts with reference answers
EVAL_PROMPTS = [
    {
        'id': 1,
        'question':  'What is QLoRA?',
        'reference': 'QLoRA fine-tunes a quantised base model using low-rank adapters, reducing GPU memory while preserving quality.',
    },
    {
        'id': 2,
        'question':  'What does GGUF stand for and why is it used?',
        'reference': 'GGUF (GPT-Generated Unified Format) is a binary format for LLM weights used by llama.cpp and Ollama for efficient CPU/GPU inference.',
    },
    {
        'id': 3,
        'question':  'How does Ollama serve a custom fine-tuned model?',
        'reference': 'Write a Modelfile pointing to the GGUF file, define a system prompt, then run ollama create and ollama run.',
    },
]


def build_judge_prompt(question: str, reference: str, model_output: str) -> str:
    return (
        f'You are evaluating an LLM response on a scale of 1 to 5.\n\n'
        f'Question: {question}\n'
        f'Reference answer: {reference}\n'
        f'Model output: {model_output}\n\n'
        f'Criteria: Relevance (does it answer the question?), '
        f'Accuracy (is it factually correct?), '
        f'Completeness (does it cover the key points?).\n\n'
        f'Respond with ONLY a single integer from 1 to 5.'
    )


def mock_llm_judge(prompt: str) -> int:
    """Simulate an LLM judge scoring 1-5. Production: call client.chat.completions.create."""
    # Simulate a slightly biased judge favouring longer, more detailed answers
    return random.choices([3, 4, 5], weights=[0.2, 0.5, 0.3])[0]


# Simulate responses from base vs fine-tuned model
def mock_model_response(question: str, model_variant: str) -> str:
    """Generate a fake model response — fine-tuned is slightly better."""
    responses = {
        'base': [
            'QLoRA is a fine-tuning method.',
            'GGUF is a file format for models.',
            'You can use ollama to serve models.',
        ],
        'ft': [
            'QLoRA (Quantized LoRA) fine-tunes a 4-bit quantized model using low-rank adapters, enabling training on a single GPU.',
            'GGUF (GPT-Generated Unified Format) is the binary model format used by llama.cpp and Ollama for fast quantized inference on CPU and GPU.',
            'To serve a custom model in Ollama: write a Modelfile with FROM pointing to your GGUF file and a SYSTEM prompt, then run ollama create <name> -f Modelfile and ollama run <name>.',
        ],
    }
    idx = EVAL_PROMPTS.index(next(p for p in EVAL_PROMPTS if p['question'] == question))
    return responses[model_variant][idx]


print(f'{"ID"} {"Model":<12} {"Score"} {"Question"}')
print('-' * 70)

for variant in ('base', 'ft'):
    for ep in EVAL_PROMPTS:
        output = mock_model_response(ep['question'], variant)
        judge_prompt = build_judge_prompt(ep['question'], ep['reference'], output)
        score = mock_llm_judge(judge_prompt)
        print(f'{ep["id"]:>2} {variant:<12} {score:>5}  {ep["question"][:45]}')


**What just happened?**

- `build_judge_prompt` constructs a G-Eval style prompt with question, reference, output, and explicit scoring criteria.
- In production, call `client.chat.completions.create(model='gpt-4o-mini', ...)` and parse the integer from the response.
- **Consistency tip:** run each prompt through the judge 3 times and average scores — single-shot LLM judgements have high variance.
- The fine-tuned model's richer responses score higher — this is what we measure in Day 10's capstone.


## Step 5 · Run 20-Prompt Quality Evaluation

Scale the evaluator to 20 prompts — the same number used in the Day 10 capstone benchmark.


In [ ]:
import random, statistics

# Generate 20 synthetic prompts for the evaluation run
TOPICS = [
    'QLoRA fine-tuning', 'GGUF conversion', 'Ollama Modelfile',
    'KV cache', 'attention mechanism', 'tokenisation', 'PEFT adapters',
    'sliding window attention', 'grouped query attention', 'speculative decoding',
    'beam search vs greedy', 'temperature sampling', 'top-p nucleus sampling',
    'BitsAndBytes quantisation', 'SFTTrainer setup', 'LoRA rank selection',
    'Alpaca dataset format', 'safetensors format', 'VRAM requirements',
    'benchmark metrics definition',
]

assert len(TOPICS) == 20, 'Need exactly 20 topics'

def run_quality_eval(model_variant: str, boost: float = 0.0) -> dict:
    """Score 20 prompts with LLM judge. boost simulates fine-tuning quality gain."""
    scores = []
    for topic in TOPICS:
        # Simulate: fine-tuned model scores 0.5-1.0 higher on domain topics
        base_score = random.choices([2, 3, 4, 5], weights=[0.05, 0.3, 0.45, 0.2])[0]
        adjusted = min(5, base_score + boost * random.uniform(0.5, 1.0))
        scores.append(round(adjusted))
    return {
        'model':     model_variant,
        'n_prompts': len(TOPICS),
        'avg_score': round(statistics.mean(scores), 2),
        'std_dev':   round(statistics.stdev(scores), 2),
        'pct_ge4':   round(sum(1 for s in scores if s >= 4) / len(scores) * 100, 1),
        'pct_le2':   round(sum(1 for s in scores if s <= 2) / len(scores) * 100, 1),
        'scores':    scores,
    }


base_eval = run_quality_eval('base',       boost=0.0)
ft_eval   = run_quality_eval('fine-tuned', boost=0.8)  # fine-tuned model scores higher

print('Quality evaluation — 20 held-out prompts')
print(f'{"Metric":<20} {"Base":>10} {"Fine-Tuned":>12} {"Delta":>8}')
print('-' * 55)
for key in ('avg_score', 'std_dev', 'pct_ge4', 'pct_le2'):
    b, f = base_eval[key], ft_eval[key]
    delta = f - b
    sign  = '+' if delta > 0 else ''
    print(f'{key:<20} {b:>10} {f:>12} {sign}{delta:>7.2f}')

print(f'\nScore distribution (base)  : {base_eval["scores"]}')
print(f'Score distribution (ft)    : {ft_eval["scores"]}')


**What just happened?**

- We evaluated 20 domain-specific prompts and computed avg score, std dev, and the fraction scoring 4+.
- `pct_ge4` is the key business metric: the proportion of responses good enough for production.
- `pct_le2` tracks failure rate — responses so bad they'd damage user trust.
- **Watch the std_dev:** a fine-tuned model with high average but also high variance may be unreliable.


## Step 6 · Compare Speed + Quality Dimensions

A good model selection decision requires looking at **both axes simultaneously**.
Plot model variants in a 2D space: quality (x-axis) vs. speed (y-axis).


In [ ]:
# Combine speed and quality results into a comparison table
comparison = [
    {
        'model':         results[0].model,
        'avg_tps':       results[0].avg_decode_tps,
        'peak_ram_gb':   results[0].peak_ram_gb,
        'avg_quality':   base_eval['avg_score'],
        'pct_ge4':       base_eval['pct_ge4'],
    },
    {
        'model':         results[1].model,
        'avg_tps':       results[1].avg_decode_tps,
        'peak_ram_gb':   results[1].peak_ram_gb,
        'avg_quality':   base_eval['avg_score'],   # same weights as base, just quantised
        'pct_ge4':       base_eval['pct_ge4'],
    },
    {
        'model':         results[2].model,
        'avg_tps':       results[2].avg_decode_tps,
        'peak_ram_gb':   results[2].peak_ram_gb,
        'avg_quality':   ft_eval['avg_score'],
        'pct_ge4':       ft_eval['pct_ge4'],
    },
]

print('Combined speed + quality comparison:')
print(f'{"Model":<25} {"Tok/s":>7} {"RAM GB":>8} {"Avg Quality":>13} {"% Score 4+":>11}')
print('-' * 70)
for c in comparison:
    print(f'{c["model"]:<25} {c["avg_tps"]:>7.1f} {c["peak_ram_gb"]:>8.2f} '
          f'{c["avg_quality"]:>13.2f} {c["pct_ge4"]:>11.1f}%')

# Compute a simple composite score: quality * speed_normalised
max_tps = max(c['avg_tps'] for c in comparison)
print('\nComposite score (quality * normalised_speed):')
for c in comparison:
    composite = c['avg_quality'] * (c['avg_tps'] / max_tps)
    print(f'  {c["model"]:<25}: {composite:.2f}')


**What just happened?**

- The composite score `quality * (tps / max_tps)` ranks models on the Pareto frontier of speed × quality.
- **Base unquantised:** lowest speed, lower quality on domain tasks (not specialised).
- **Q4 quantised base:** faster (less memory bandwidth), same quality as base (same weights).
- **Fine-tuned Q4:** same speed as base Q4, higher quality — best overall composite score.
- This is the expected outcome of the Day 10 capstone.


## Step 7 · Write the Benchmark Report

A one-page benchmark report should answer three questions:
1. **Which model wins on speed?** (and by how much)
2. **Which model wins on quality?** (and by how much)
3. **Which model to deploy** for this specific use case?


In [ ]:
import datetime

def generate_benchmark_report(comparison: list[dict], eval_base: dict, eval_ft: dict) -> str:
    """Generate a structured one-page benchmark report as markdown."""
    fastest    = max(comparison, key=lambda c: c['avg_tps'])
    best_qual  = max(comparison, key=lambda c: c['avg_quality'])
    best_comp  = max(comparison, key=lambda c: c['avg_quality'] * (c['avg_tps'] / max(x['avg_tps'] for x in comparison)))

    lines = [
        f'# Llama-3.2-1B Benchmark Report',
        f'Generated: {datetime.date.today()}',
        '',
        '## Setup',
        '- Hardware: CPU inference (simulate GPU: multiply tok/s by ~6)',
        '- Quantisation: Q4_K_M (GGUF via llama.cpp)',
        f'- Quality eval: {eval_base["n_prompts"]} prompts, LLM judge (1-5 scale)',
        '',
        '## Speed Results',
        f'| Model | Tok/s | Peak RAM |',
        f'|---|---|---|',
    ]
    for c in comparison:
        lines.append(f'| {c["model"]} | {c["avg_tps"]} | {c["peak_ram_gb"]} GB |')

    lines += [
        '',
        '## Quality Results',
        f'| Model | Avg Score | % Score 4+ | % Score 1-2 |',
        f'|---|---|---|---|',
        f'| base | {eval_base["avg_score"]} | {eval_base["pct_ge4"]}% | {eval_base["pct_le2"]}% |',
        f'| fine-tuned | {eval_ft["avg_score"]} | {eval_ft["pct_ge4"]}% | {eval_ft["pct_le2"]}% |',
        '',
        '## Findings',
        f'- **Fastest model:** {fastest["model"]} at {fastest["avg_tps"]} tok/s',
        f'- **Highest quality:** {best_qual["model"]} with avg score {best_qual["avg_quality"]}',
        f'- **Best overall:** {best_comp["model"]} (quality x speed composite)',
        '',
        '## Recommendation',
        '- **Deploy:** fine-tuned Q4 model — best composite, domain-adapted weights, minimal RAM.',
        '- **When to use base:** general-purpose tasks outside the training domain.',
        '- **Next step:** run lm-evaluation-harness on ARC/HellaSwag to confirm no catastrophic forgetting.',
    ]
    return '\n'.join(lines)


report = generate_benchmark_report(comparison, base_eval, ft_eval)
print(report)


**What just happened?**

- `generate_benchmark_report` produces a structured markdown document you can paste into a wiki or PR description.
- The report answers all three questions: speed winner, quality winner, deployment recommendation.
- **Catastrophic forgetting** is a real risk: the fine-tuned model might score worse on general benchmarks (ARC, HellaSwag) even while scoring better on domain prompts.
- Always run a general-purpose benchmark alongside your domain eval to detect regression.


In [ ]:
# Challenge: Extend the benchmarker to detect and report catastrophic forgetting.
#
# Catastrophic forgetting occurs when a fine-tuned model scores lower on general tasks
# even while improving on domain tasks.
#
# Requirements:
#   1. Define two prompt sets: DOMAIN_PROMPTS (your fine-tuning domain) and GENERAL_PROMPTS
#   2. Run the LLM judge on both sets for both base and fine-tuned models
#   3. Compute: domain_gain = ft_domain_avg - base_domain_avg
#              general_delta = ft_general_avg - base_general_avg
#   4. Flag 'FORGETTING DETECTED' if general_delta < -0.3 (3+ point drop on 1-5 scale)
#   5. Print a summary table with all four scores and the forgetting flag
#
# Scaffold:
DOMAIN_PROMPTS_CHALLENGE  = [f'Domain question {i}' for i in range(1, 11)]
GENERAL_PROMPTS_CHALLENGE = [f'General question {i}' for i in range(1, 11)]

def detect_forgetting(base_domain_avg, base_general_avg, ft_domain_avg, ft_general_avg):
    # TODO: compute domain_gain and general_delta
    # TODO: print summary table
    # TODO: print FORGETTING DETECTED if general_delta < -0.3
    pass

# detect_forgetting(base_domain_avg=3.2, base_general_avg=3.5, ft_domain_avg=4.1, ft_general_avg=3.1)
print('Implement detect_forgetting() above, then call it with your eval results.')


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| Prefill vs decode | Prefill is parallel and fast; decode is sequential — measure tok/s at decode only |
| Q4 memory savings | ~0.5 GB per billion params — a 1B model fits in under 1 GB RAM |
| LLM judge | G-Eval style: question + reference + output + criteria → score 1-5; average 3 runs |
| Catastrophic forgetting | Fine-tuning can hurt general-task scores; always run ARC/HellaSwag alongside domain eval |
| Composite score | quality × (speed / max_speed) ranks models on the Pareto frontier |

> **Tip:** Measure tokens/sec at the decode phase, not prefill. If you see slow inference, check whether you are bottlenecked on prefill (context length) or decode (generation speed).

---
## What's next
**Day 10** → Capstone — fine-tune Llama-3.2-1B with QLoRA, convert to GGUF, serve via Ollama, and run the full benchmark end-to-end.

Mark Day 9 complete in your [tracker](../index.html).
